# DATA09 v0-flow 双通道TDMS特征提取

本 notebook 用于对 `E:\codes\ZZ-BK\DATA09\v0-flow` 目录下的连续数据进行特征提取。

## 数据特点
- 文件格式：TDMS双通道文件
- 数据路径：`E:\codes\ZZ-BK\DATA09\v0-flow`
- 需要读取指定通道的数据进行特征提取


In [2]:
from __future__ import annotations

import os
import sys
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# 1 = prefer CUDA when available; 0/false/off/no = force CPU.
os.environ.setdefault('FEA_CPT_USE_GPU', '1')

workspace = Path.cwd()
if not (workspace / 'src').exists():
    workspace = workspace.parent
if str(workspace / 'src') not in sys.path:
    sys.path.insert(0, str(workspace / 'src'))

from fea_cpt_gpu_v2_0.sliding_window import (
    SlidingWindowConfig,
    build_sliding_window_dataset,
    compute_shared_stft,
    discover_source_files,
    downsample_source_files,
    gpu_backend_info,
    list_window_ranges,
    upsample_to_target,
    process_source_file,
    _auto_detect_workers,
)
from fea_cpt_gpu_v2_0.params import DEFAULT_FEATURE_PARAMS

print(f'workspace = {workspace}')
print(gpu_backend_info())
print(f'Python = {sys.version}')
print(f'CPU 核心数: {os.cpu_count()}')
print(f'推荐 workers: {_auto_detect_workers()}')
print('v2.0 所有模块加载成功')


workspace = e:\codes\ZZ-BK
[GPU] 使用 CUDA: NVIDIA GeForce RTX 4050 Laptop GPU
Python = 3.12.13 | packaged by conda-forge | (main, Mar  5 2026, 16:36:12) [MSC v.1944 64 bit (AMD64)]
CPU 核心数: 16
推荐 workers: 14
v2.0 所有模块加载成功


In [3]:
# =========================
# 全局配置
# =========================

# 每个输入路径单独设置随机抽取比例。ratio=1.0 表示处理该路径下全部文件。
# 示例：
# RAW_DATA_ROOT_SPECS = [
#     (r"E:\codes\ZZ-BK\DATA09\v0-flow", 0.30),
#     (r"E:\codes\ZZ-BK\DATA09\v0-flow-extra", 0.80),
# ]
RAW_DATA_ROOT_SPECS = [
    
    (r"K:\0904-bk-v0", 1.0),(r"K:\0829-bk-v0", 1.0),
    # (r"E:\PCCP\20260902挖掘机17 16号管v0", 0.1),
    # (r"E:\PCCP\20260902挖掘机埋图v0", 0.01)
]


def parse_input_specs(
    specs: list[tuple[str | Path, float]] | tuple[tuple[str | Path, float], ...],
) -> list[tuple[Path, float]]:
    parsed: list[tuple[Path, float]] = []
    seen: set[str] = set()
    for raw_path, ratio in specs:
        path = Path(raw_path).expanduser()
        key = str(path.resolve()) if path.exists() else str(path)
        if key in seen:
            continue
        ratio = float(ratio)
        if not 0.0 < ratio <= 1.0:
            raise ValueError(f'抽取比例必须在 (0, 1] 内: {raw_path} -> {ratio}')
        parsed.append((path, ratio))
        seen.add(key)
    if not parsed:
        raise ValueError('RAW_DATA_ROOT_SPECS 为空，请至少配置一个输入路径')
    return parsed

RAW_DATA_SPECS = parse_input_specs(RAW_DATA_ROOT_SPECS)
RAW_DATA_ROOTS = [path for path, _ratio in RAW_DATA_SPECS]

WINDOW_DURATION_S = 0.03
WINDOW_OVERLAP = 0.5
assert 0.0 <= WINDOW_OVERLAP < 1.0

TARGET_SAMPLE_RATE = 1000_000.0
PREPROC_BAND = (1_000.0, 95_000.0)

BANDS = [
    ('b_1k_100k',  (1_000.0,  100_000.0)),
    ('b_1k_10k',   (1_000.0,  10_000.0)),
    ('b_10k_20k',  (10_000.0, 20_000.0)),
    ('b_20k_30k',  (20_000.0, 30_000.0)),
    ('b_30k_40k',  (30_000.0, 40_000.0)),
    ('b_40k_60k', (40_000.0, 60_000.0)),
    ('b_10k_50k',  (10_000.0, 50_000.0)),
    ('b_1k_50k', (1_000.0, 50_000.0)),
]

# v2.0 并行设置
WINDOW_WORKERS = None          # None = 自动检测
WINDOW_BATCH_SIZE = 2048       # 回退路径用
ENABLE_NUMA_BINDING = True     # NUMA 绑定
ENABLE_SHARED_STFT = True      # 跨频带共享 STFT；没有 CUDA 时自动使用 CPU STFT
STFT_BATCH_SIZE = 800          # v2.0: GPU STFT 每批窗口数（控制 GPU 内存）

# 文件抽样设置
ENABLE_FILE_SAMPLING = True    # True = 按 RAW_DATA_ROOT_SPECS 中每个路径的 ratio 随机抽样
FILE_SAMPLE_SEED = 42          # 随机种子（保证可复现）

RUN_TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
OUTPUT_ROOT = workspace / 'outputs' / 'DATA09_v0-flow_features-bk-test'
RUN_OUTPUT_ROOT = OUTPUT_ROOT / f'run_{RUN_TIMESTAMP}'
PROCESS_LOG_DIR = OUTPUT_ROOT / '_process_logs'
PROCESSED_LIST_PATH = PROCESS_LOG_DIR / 'processed_source_files_v0.txt'
NPZ_PER_CSV = 100
MAX_FILES: int | None = None
TDMS_FALLBACK_SAMPLE_RATE_HZ: float | None = 1_000_000.0  # 1MHz 采样率

# 双通道TDMS设置
TARGET_CHANNEL_NAME = 'Untitled'  # 指定要读取的通道名称（实际通道: Untitled / Untitled 1）

RUN_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
PROCESS_LOG_DIR.mkdir(parents=True, exist_ok=True)

auto_workers = _auto_detect_workers() if WINDOW_WORKERS is None else WINDOW_WORKERS
print(f'数据路径: {len(RAW_DATA_SPECS)} 个根目录/文件')
for root, ratio in RAW_DATA_SPECS:
    print(f'  - {root} (抽取比例={ratio:.3g})')
print(f'滑窗: {WINDOW_DURATION_S*1000:.0f}ms, 重叠 {WINDOW_OVERLAP*100:.0f}%')
print(f'目标采样率: {TARGET_SAMPLE_RATE/1000:.0f} kHz')
print(f'频带数: {len(BANDS)}')
print(f'并行: {auto_workers} workers, STFT batch={STFT_BATCH_SIZE}')
print(f'NUMA 绑定: {ENABLE_NUMA_BINDING}')
print(f'共享 STFT: {ENABLE_SHARED_STFT}')
print(f'文件抽样: {ENABLE_FILE_SAMPLING} (种子={FILE_SAMPLE_SEED})')
print(f'输出目录: {RUN_OUTPUT_ROOT}')
print(f'断点日志: {PROCESSED_LIST_PATH}')
print(f'指定通道: {TARGET_CHANNEL_NAME}')

config = SlidingWindowConfig(
    bands=BANDS,
    preproc_band=PREPROC_BAND,
    window_duration_s=WINDOW_DURATION_S,
    window_overlap=WINDOW_OVERLAP,
    target_sample_rate=TARGET_SAMPLE_RATE,
    tdms_fallback_sample_rate=TDMS_FALLBACK_SAMPLE_RATE_HZ,
    tdms_channel_name=TARGET_CHANNEL_NAME,
    window_workers=WINDOW_WORKERS,
    window_batch_size=WINDOW_BATCH_SIZE,
    enable_numa_binding=ENABLE_NUMA_BINDING,
    enable_shared_stft=ENABLE_SHARED_STFT,
    stft_batch_size=STFT_BATCH_SIZE,
)
print('\n配置对象创建成功')


数据路径: 2 个根目录/文件
  - K:\0904-bk-v0 (抽取比例=1)
  - K:\0829-bk-v0 (抽取比例=1)
滑窗: 30ms, 重叠 50%
目标采样率: 1000 kHz
频带数: 8
并行: 14 workers, STFT batch=800
NUMA 绑定: True
共享 STFT: True
文件抽样: True (种子=42)
输出目录: e:\codes\ZZ-BK\outputs\DATA09_v0-flow_features-bk-test\run_20260906_011739
断点日志: e:\codes\ZZ-BK\outputs\DATA09_v0-flow_features-bk-test\_process_logs\processed_source_files_v0.txt
指定通道: Untitled

配置对象创建成功


In [4]:
# =========================
# 数据文件发现与按路径抽样
# =========================

import random
from collections import Counter

def sample_files_for_root(files: list[Path], ratio: float, seed: int, root_index: int) -> list[Path]:
    if ratio >= 1.0:
        return list(files)
    if not files:
        return []
    sample_count = max(1, int(round(len(files) * ratio)))
    sample_count = min(sample_count, len(files))
    rng = random.Random(seed + root_index)
    return sorted(rng.sample(list(files), sample_count))

source_files_all: list[Path] = []
source_files_sampled: list[Path] = []
seen_files: set[str] = set()

print('按输入路径发现文件并抽样:')
for root_index, (root, ratio) in enumerate(RAW_DATA_SPECS):
    root_files = discover_source_files([root], max_files=MAX_FILES)
    sampled_files = sample_files_for_root(root_files, ratio if ENABLE_FILE_SAMPLING else 1.0, FILE_SAMPLE_SEED, root_index)

    print(
        f'  [{root_index + 1}] {root}: '
        f'发现 {len(root_files)} 个, 抽取 {len(sampled_files)} 个, 比例={ratio:.3g}'
    )

    source_files_all.extend(root_files)
    for file_path in sampled_files:
        file_key = str(file_path)
        if file_key in seen_files:
            continue
        source_files_sampled.append(file_path)
        seen_files.add(file_key)

source_files_all = sorted(dict.fromkeys(source_files_all))
source_files = sorted(source_files_sampled)

if not source_files_all:
    raise FileNotFoundError(f'未找到任何 .npz/.tdms 文件，请检查路径: {RAW_DATA_ROOTS}')

if not source_files:
    raise FileNotFoundError('抽样后没有任何待处理文件，请检查各路径抽取比例和输入路径')

if PROCESSED_LIST_PATH.exists():
    processed_set = {
        line.strip()
        for line in PROCESSED_LIST_PATH.read_text(encoding='utf-8').splitlines()
        if line.strip()
    }
else:
    processed_set = set()

source_files_before_resume = list(source_files)
source_files = [f for f in source_files if str(f) not in processed_set]

folder_counts_all = Counter(f.parent.name for f in source_files_all)
folder_counts_sampled = Counter(f.parent.name for f in source_files_before_resume)
folder_counts_pending = Counter(f.parent.name for f in source_files)

print(f'\n总发现文件: {len(source_files_all)}')
print(f'抽样后文件: {len(source_files_before_resume)}')
print(f'断点日志中已有 {len(processed_set)} 个已处理文件')
print(f'本次待处理 {len(source_files)} 个源文件')

print(f'\n各文件夹文件数:')
for folder, total_count in sorted(folder_counts_all.items()):
    sampled_count = folder_counts_sampled.get(folder, 0)
    pending_count = folder_counts_pending.get(folder, 0)
    print(f'  {folder}: {pending_count} 待处理 / {sampled_count} 抽样 / {total_count} 总数')

npz_count = sum(1 for f in source_files_all if f.suffix.lower() == '.npz')
tdms_count = sum(1 for f in source_files_all if f.suffix.lower() == '.tdms')
print(f'\n文件格式: {npz_count} npz, {tdms_count} tdms')


按输入路径发现文件并抽样:
  [1] K:\0904-bk-v0: 发现 91 个, 抽取 91 个, 比例=1
  [2] K:\0829-bk-v0: 发现 228 个, 抽取 228 个, 比例=1

总发现文件: 319
抽样后文件: 319
断点日志中已有 0 个已处理文件
本次待处理 319 个源文件

各文件夹文件数:
  0829-bk-v0: 228 待处理 / 228 抽样 / 228 总数
  0904-bk-v0: 91 待处理 / 91 抽样 / 91 总数

文件格式: 0 npz, 319 tdms


In [5]:
# =========================
# 查看TDMS文件结构
# =========================

from nptdms import TdmsFile

if source_files:
    test_file = source_files[0]
    print(f'查看文件结构: {test_file.name}')
    
    td = TdmsFile.read(test_file)
    
    print(f'\n Groups:')
    for g in td.groups():
        print(f'  {g.name}')
        for c in g.channels():
            print(f'    Channel: {c.name}')
            print(f'    Length: {len(c[:])}')
            print(f'    Properties: {dict(c.properties)}')
            print()


查看文件结构: SemiPhase-1MHz-2026-8-29-15-10-14.tdms

 Groups:
  Data
    Channel: Untitled
    Length: 6000000
    Properties: {'NI_ArrayColumn': 0}

    Channel: Untitled 1
    Length: 6000000
    Properties: {'NI_ArrayColumn': 1}



In [6]:
# =========================
# 单文件处理测试
# =========================

import time

test_file = source_files[0]
print(f'测试文件: {test_file.name}')

from fea_cpt_gpu_v2_0.sliding_window import load_source_file, upsample_to_target

# 使用load_source_file加载指定通道（通过config.tdms_channel_name参数）
src = load_source_file(test_file, config.tdms_fallback_sample_rate, config.tdms_channel_name)
raw = np.asarray(src['signal_values'], dtype=float)
orig_rate = float(src['sample_rate'])
print(f'  原始采样率: {orig_rate/1000:.0f} kHz, 样本数: {len(raw):,}')
print(f'  通道: {src["source_channel_name"]}')

sig_up, eff_rate = upsample_to_target(raw, orig_rate, config.target_sample_rate)
print(f'  升采样后: {eff_rate/1000:.0f} kHz, 样本数: {len(sig_up):,}')

windows = list_window_ranges(len(sig_up), eff_rate, config.window_duration_s, config.window_overlap)
print(f'  窗口数: {len(windows)}')

print('\n开始单文件特征计算（v2.0: 持久化进程池 + 共享内存 + GPU集中化STFT）...')
t0 = time.time()
df_feat, df_log = process_source_file(test_file, config)
elapsed = time.time() - t0

print(f'  耗时: {elapsed:.1f} s')
print(f'  特征表形状: {df_feat.shape}')
print('\n单文件测试通过！')


测试文件: SemiPhase-1MHz-2026-8-29-15-10-14.tdms
  原始采样率: 1000 kHz, 样本数: 6,000,000
  通道: Untitled
  升采样后: 1000 kHz, 样本数: 6,000,000
  窗口数: 399

开始单文件特征计算（v2.0: 持久化进程池 + 共享内存 + GPU集中化STFT）...
  耗时: 363.9 s
  特征表形状: (399, 661)

单文件测试通过！


In [7]:
# =========================
# Batch processing
# =========================

import time
from datetime import datetime

processed_list_path = PROCESSED_LIST_PATH

print(f'{"="*60}')
print('Batch processing start (v2.0 pipeline)')
print(f'Time: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print(f'File count: {len(source_files)}')
print('Progress bar covers all pending files from all input folders; ETA is estimated from completed files.')
print(f'Output dir: {RUN_OUTPUT_ROOT}')
print(f'Processed log: {processed_list_path}')
print(f'{"="*60}')

t_start = time.time()

stats = build_sliding_window_dataset(
    source_paths=source_files,
    config=config,
    output_dir=RUN_OUTPUT_ROOT,
    processed_list_path=processed_list_path,
    npz_per_csv=NPZ_PER_CSV,
    show_progress=True,
)

t_elapsed = time.time() - t_start

# Add creation time to generated result filenames. RUN_OUTPUT_ROOT is unique per run,
# while processed_list_path is stable across runs for resume/skip behavior.
for csv_path in sorted(RUN_OUTPUT_ROOT.glob('features_part_*.csv')):
    csv_path.rename(csv_path.with_name(f'features_{RUN_TIMESTAMP}_{csv_path.name.removeprefix("features_")}'))
for csv_path in sorted(RUN_OUTPUT_ROOT.glob('log_part_*.csv')):
    csv_path.rename(csv_path.with_name(f'log_{RUN_TIMESTAMP}_{csv_path.name.removeprefix("log_")}'))

print()
print(f'{"="*60}')
print('Batch processing complete')
print(f'Elapsed: {t_elapsed:.1f} s ({t_elapsed/60:.1f} min)')
print(f'Processed files: {stats["processed"]}')
print(f'Skipped files: {stats["skipped"]}')
print(f'Failed files: {stats["failed"]}')
print(f'Total windows: {stats["windows"]}')
if stats["processed"] > 0:
    avg = t_elapsed / stats["processed"]
    print(f'Average per file: {avg:.1f} s')
print(f'断点日志保留在: {processed_list_path}')
print(f'{"="*60}')



Batch processing start (v2.0 pipeline)
Time: 2026-09-06 01:23:43
File count: 319
Progress bar covers all pending files from all input folders; ETA is estimated from completed files.
Output dir: e:\codes\ZZ-BK\outputs\DATA09_v0-flow_features-bk-test\run_20260906_011739
Processed log: e:\codes\ZZ-BK\outputs\DATA09_v0-flow_features-bk-test\_process_logs\processed_source_files_v0.txt


处理全部文件:   0%|          | 0/319 [00:00<?, ?file/s]

PermissionError: [Errno 13] Permission denied: 'e:\\codes\\ZZ-BK\\outputs\\DATA09_v0-flow_features-bk-test\\run_20260906_011739\\features_part_0001.csv'

In [ ]:
# =========================
# 处理结果汇总
# =========================

feature_chunks = sorted(RUN_OUTPUT_ROOT.glob(f'features_{RUN_TIMESTAMP}_part_*.csv'))
log_chunks = sorted(RUN_OUTPUT_ROOT.glob(f'log_{RUN_TIMESTAMP}_part_*.csv'))

print(f'特征 CSV 文件数: {len(feature_chunks)}')

if feature_chunks:
    df_sample = pd.read_csv(feature_chunks[0], nrows=5)
    print(f'特征 CSV 列数: {len(df_sample.columns)}')

    total_windows = 0
    total_files_set = set()
    for chunk in feature_chunks:
        df = pd.read_csv(chunk, usecols=['source_file_name', 'window_id'])
        total_windows += len(df)
        total_files_set.update(df['source_file_name'].unique())
    print(f'总窗口数: {total_windows:,}')
    print(f'本次输出总文件数: {len(total_files_set)}')


In [ ]:
# =========================
# 特征质量检查
# =========================

if feature_chunks:
    df_check = pd.read_csv(feature_chunks[0])

    meta_cols = {
        'source_file_name', 'source_file_path', 'source_format',
        'source_group_name', 'source_channel_name', 'source_detail',
        'window_id', 'window_start_index', 'window_end_index',
        'window_length_samples', 'window_step_samples',
        'window_duration_s', 'window_start_offset_s',
        'sample_rate_hz', 'original_sample_rate_hz',
        'source_n_samples', 'source_duration_s',
        'starttime_raw', 'arrival_time_raw', 'sample_type',
        'window_start_datetime',
    }
    feature_cols = [c for c in df_check.columns if c not in meta_cols]

    print(f'特征列数: {len(feature_cols)}')

    full_nan = [c for c in feature_cols if pd.to_numeric(df_check[c], errors='coerce').isna().all()]
    if full_nan:
        print(f'警告: {len(full_nan)} 个特征全为 NaN')
    else:
        print('无全 NaN 特征，数据质量良好')

    # 快速统计
    nan_stats = []
    for col in feature_cols[:20]:
        series = pd.to_numeric(df_check[col], errors='coerce')
        nan_ratio = series.isna().sum() / len(series)
        nan_stats.append({'feature': col, 'nan_ratio': nan_ratio, 'mean': series.mean()})
    print('\n前20个特征统计:')
    print(pd.DataFrame(nan_stats).to_string(index=False))


In [ ]:
# =========================
# 运行日志
# =========================

processed_log = PROCESSED_LIST_PATH
failed_log = RUN_OUTPUT_ROOT / 'failed_samples.log'

if processed_log.exists():
    lines = [line for line in processed_log.read_text(encoding='utf-8').splitlines() if line.strip()]
    print(f'累计已处理文件记录: {len(lines)} 条')
    print('\n最近处理的 5 个文件:')
    for line in lines[-5:]:
        print(f'  {Path(line).name}')
else:
    print('暂无已处理文件记录')

if failed_log.exists():
    print(f'\n本次失败样本日志:')
    print(failed_log.read_text(encoding='utf-8'))
else:
    print('\n本次无失败样本')


## 架构说明

```
主进程(GPU):
  加载文件N → 批量GPU STFT(200窗口/批) → 写入共享内存 → 加载文件N+1 → ...
       ↕ SharedMemory (signal_pre + STFT结果)
  预加载线程: 后台加载下一文件，与当前文件的特征计算并行

持久Worker1-14(CPU):
  从共享内存读取信号切片 + STFT切片 → butter_filter + ridge + ISTFT + wavelet + 特征计算 → 返回结果
```

**关键保证：所有优化不改变特征计算公式，仅改变计算调度方式。**

### 双通道TDMS处理说明

本notebook针对双通道TDMS文件进行了适配：
1. 通过 `SlidingWindowConfig.tdms_channel_name` 参数指定要读取的通道名称
2. 在notebook中设置 `TARGET_CHANNEL_NAME = 'Untitled'` 指定通道
3. 如果指定通道不存在，会回退到默认的通道选择逻辑
4. 相关修改已集成到 `fea_cpt_gpu_v2_0.sliding_window` 模块中
5. 没有 CUDA 时自动使用 CPU STFT；断点日志固定保存在 `_process_logs/processed_source_files.txt`
